# Projeto 7: Classificação multiclasse iris com validação cruzada

## Etapa 1: Importação das bibliotecas

In [2]:
# !pip install skorch para fazer a validação cruzada, no ambiente do anaconda YouTube

# no linux
# conda env list
# conda activate YouTube


In [3]:
import pandas as pd
import numpy as np
import torch.nn as nn
from skorch import NeuralNetClassifier
import torch
from sklearn.model_selection import cross_val_score
torch.__version__

'2.8.0+cu128'

## Etapa 2: Base de dados

servem para garantir a reprodutibilidade.

Em muitos algoritmos de machine learning e deep learning, a aleatoriedade é usada para tarefas como a inicialização dos pesos da rede ou a divisão dos dados de forma aleatória.

Ao usar a <b>função seed</b> com o mesmo número (neste caso, 123), garantimos que as operações aleatórias do NumPy e do PyTorch produzirão sempre a mesma sequência de números. Isso significa que, se nós ou outras pessoas rodarem o código novamente, os resultados serão exatamente os mesmos, o que é crucial para depurar e validar experimentos científicos e de engenharia.

In [4]:
# semente aleatória para o início dos pesos
np.random.seed(123)
torch.manual_seed(123)

In [5]:
base = pd.read_csv('iris.csv')
previsores = base.iloc[:, 0:4].values
classe = base.iloc[:, 4].values

In [29]:
# valores categoricos em strings para transformar em números
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
classe = encoder.fit_transform(classe)

In [7]:
base.tail(5)

,sepal length,sepal width,petal length,petal width,class
145,6.7,3.0,5.2,2.3,Iris-virginica
146,6.3,2.5,5.0,1.9,Iris-virginica
147,6.5,3.0,5.2,2.0,Iris-virginica
148,6.2,3.4,5.4,2.3,Iris-virginica
149,5.9,3.0,5.1,1.8,Iris-virginica


In [8]:
base['class'].unique()

array(['Iris-setosa', 'Iris-versicolor', 'Iris-virginica'], dtype=object)

In [9]:
np.unique(classe)

array([0, 1, 2])

In [10]:
previsores = previsores.astype('float32')
classe = classe.astype('int64')

In [14]:
previsores[:4]

array([[5.1, 3.5, 1.4, 0.2],
       [4.9, 3. , 1.4, 0.2],
       [4.7, 3.2, 1.3, 0.2],
       [4.6, 3.1, 1.5, 0.2]], dtype=float32)

In [15]:
classe[:4]

array([0, 0, 0, 0])

## Etapa 3: Construção do modelo

In [16]:
# aumentamos os neurônios para comparar depois os resultados 
# com 16 neurônios nas duas camadas ocultas

In [23]:
class classificador_torch(nn.Module):
    # arquiteura _init_
    def __init__(self):
        super().__init__()
        # camada densa
        # Uma camada de entrada (self.dense0) que recebe 4 dados.
        # Duas camadas ocultas (self.dense0 e self.dense1), cada uma com 16 neurônios.       
        self.dense0 = nn.Linear(4, 16)
        # camada de ativação ReLu
        #Funções de ativação ReLU (self.activation0 e self.activation1) para adicionar não linearidade à rede.
        self.activation0 = nn.ReLU()
        # camada densa.
        self.dense1 = nn.Linear(16, 16)
        self.activation1 = nn.ReLU()
        #Uma camada de saída (self.dense2) que gera 3 resultados, o que indica um problema de classificação de 3 classes.
        self.dense2 = nn.Linear(16, 3)

    # foward para fazer as ligações entre as camadas.
    #  Fluxo de Dados (forward)
    # forward define a ordem em que os dados passam por essas camadas. É o caminho que os dados percorrem, da entrada até a saída.
    #  entrada X passa pela primeira camada densa e pela ativação.
    # O resultado passa pela segunda camada densa e por outra ativação.
    #Por fim, o resultado passa pela última camada densa para gerar a previsão final, que é então retornada.
    def forward(self, X):
        X = self.dense0(X)
        X = self.activation0(X)
        X = self.dense1(X)
        X = self.activation1(X)
        X = self.dense2(X)
        return X


###                                         Camada de ativação

$\begin{array}{|c|c|c|c|}
\hline
\textbf{Função} & \textbf{Fórmula} & \textbf{Intervalo de saída} & \textbf{Uso comum} \\
\hline
\text{Sigmoid} & 
\sigma(x) = \frac{1}{1 + e^{-x}} & (0, 1) & \text{Classificação binária} \\
\hline
\text{Tanh} & 
\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}} & (-1, 1) & \text{Camadas ocultas (alternativa ao ReLU)} \\
\hline
\text{ReLU} & 
f(x) = \max(0, x) & [0, \infty) & \text{Camadas ocultas (mais usado)} \\
\hline
\text{Leaky ReLU} & 
f(x) = \begin{cases}
x & \text{se } x > 0 \\
\alpha x & \text{se } x \leq 0
\end{cases} & (-\infty, \infty) & \text{Evita neurônios mortos no ReLU} \\
\hline
\text{ELU} & 
f(x) = \begin{cases}
x & \text{se } x > 0 \\
\alpha(e^x - 1) & \text{se } x \leq 0
\end{cases} & (-\alpha, \infty) & \text{Camadas ocultas, alternativa ao ReLU} \\
\hline
\text{Softmax} & 
f(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}} & (0, 1), \ \sum f(x_i) = 1 & \text{Classificação multiclasse} \\
\hline
\end{array}$


🔹 O ReLU (Rectified Linear Unit)

Características:

Simples, rápido de calcular.

Resolve o problema do vanishing gradient (quando os gradientes ficam quase zero em sigmoides/tanh).

Muito usado em camadas ocultas.

⚠️ Problema: pode causar “neurônios mortos” (quando muitos valores ficam ≤ 0 e nunca mais reativam).




In [20]:
# agora usamos a integração das redes neurais com o sklearn

#### Criando um wrapper, ou um invólucro, que adapta um modelo de rede neural do PyTorch (classificador_torch) para ser usado com a biblioteca scikit-learn.

Isso é feito com a biblioteca skorch, que permite combinar a flexibilidade do PyTorch (para construir redes neurais personalizadas) com a robustez do scikit-learn (para tarefas como validação cruzada, ajuste de hiperparâmetros com Grid Search, e pipelines de pré-processamento).

#### Detalhes dos Parâmetros

- <b>module = classificador_torch</b>: Este é o modelo PyTorch que foi definido. O NeuralNetClassifier o "envolve" para que ele possa ser tratado como um classificador padrão do scikit-learn.

- <b>criterion = torch.nn.CrossEntropyLoss </b>: Define a função de perda (a métrica de erro) que o modelo tentará minimizar durante o treinamento. A CrossEntropyLoss é a função de perda padrão e ideal para problemas de classificação de múltiplas classes.

- <b>optimizer = torch.optim.Adam</b>: Especifica o algoritmo de otimização que será usado para ajustar os pesos da rede. O Adam é uma escolha popular e geralmente muito eficiente.

- <b>max_epochs = 1000</b>: Define o número máximo de épocas que o modelo irá treinar. Uma época é uma passagem completa por todo o conjunto de dados de treino.

- <b>batch_size = 10</b>: Indica o tamanho dos lotes de dados que a rede neural irá processar em cada iteração do treinamento.

- <b>train_split = False</b>: Diz ao skorch para não dividir os dados de treino internamente para validação. Isso é útil quando _planejamos fazer sua própria validação cruzada_ (como nós fariamos com o scikit-learn) e não quer que o skorch divida os dados novamente.

In [21]:
classificador_sklearn = NeuralNetClassifier(module = classificador_torch,
                                            criterion = torch.nn.CrossEntropyLoss,
                                            optimizer = torch.optim.Adam,
                                            max_epochs = 1000,
                                            batch_size = 10,
                                            train_split = False)

## Etapa 4: Validação cruzada

In [24]:
resultados = cross_val_score(classificador_sklearn, previsores, classe, cv = 5,
                             scoring = 'accuracy') # faremos a validação cruzada em 5 partes. 

  epoch    train_loss     dur
-------  ------------  ------
      1        2.0070  0.0135
      2        1.0293  0.0074
      3        0.9454  0.0237
      4        0.8550  0.0222
      5        0.7365  0.0197
      6        0.6514  0.0246
      7        0.5581  0.0221
      8        0.5056  0.0243
      9        0.5162  0.0197
     10        0.4109  0.0225
     11        0.5542  0.0254
     12        0.4098  0.0243
     13        0.5332  0.0272
     14        0.4993  0.0214
     15        0.3979  0.0157
     16        0.3369  0.0143
     17        0.3687  0.0191
     18        0.3944  0.0238
     19        0.2903  0.0220
     20        0.3097  0.0220
     21        0.2867  0.0236
     22        0.2556  0.0235
     23        0.2571  0.0232
     24        0.2474  0.0207
     25        0.2309  0.0245
     26        0.2160  0.0212
     27        0.2134  0.0203
     28        0.2090  0.0246
     29        0.2031  0.0209
     30        0.1998  0.0233
     31        0.1923  0.0244
     32   

## Processo chamado validação cruzada (cross-validation) para avaliar o desempenho do seu modelo de forma robusta e confiável.

Como Funciona a Validação Cruzada

Em vez de dividir seus dados em um único conjunto de treino e um de teste, a validação cruzada divide os dados em várias partes, chamadas de "folds". O processo então treina e testa o modelo várias vezes.

Detalhes:

- A) cross_val_score(...): Esta é a função do scikit-learn que orquestra todo o processo de validação cruzada. Ela automatiza o treinamento e a avaliação do modelo.

- B) classificador_sklearn: Este é o nosso modelo criado de rede neural (do PyTorch, mas adaptado para o scikit-learn com skorch). É o modelo que a função irá treinar e testar.

- C) previsores e classe: Estes são os nossos dados completos, com os atributos (previsores) e os rótulos (classe).

- D) <b> cv = 5</b>: Este parâmetro é o mais importante. Ele define o número de "folds" (ou dobras) que os dados serão divididos. Com cv = 5, o processo será executado 5 vezes. A cada iteração:

-> Um dos 5 folds é reservado como conjunto de teste.

-> Os outros 4 folds são usados para treinar o modelo.

O modelo treinado é testado no fold de teste e a pontuação é registrada.

- E) scoring = 'accuracy': Este parâmetro especifica qual métrica de desempenho deve ser calculada em cada rodada de teste. Neste caso, é a acurácia (a porcentagem de previsões corretas).

resultados Armazena:

Ao final das 5 iterações, a variável resultados será uma lista (ou array) com <b>5 valores de acurácia</b>, um para cada rodada de teste.

Ao analisar esses 5 valores, você pode entender melhor a consistência do seu modelo. Se os resultados forem próximos (por exemplo, [0.92, 0.91, 0.93, 0.90, 0.92]), significa que o modelo é estável. Se houver muita variação, pode indicar que o modelo não está generalizando bem.

In [25]:
media = resultados.mean()
desvio = resultados.std()

In [26]:
media, desvio

(0.9733333333333334, 0.03265986323710903)

Média próxima de 1, é bom

In [27]:
resultados

array([1.        , 1.        , 0.93333333, 0.93333333, 1.        ])

In [28]:
# muito próximo, mas valores quase 1 pode indicar dados com overfitting, é necessário estudar mais a fundo.